In [ ]:
import os
import requests 
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("WB_API_KEY")
BASE_URL_ANALYTICS = "https://seller-analytics-api.wildberries.ru/api/v1"

def fetch_report(endpoint: str, params: dict):
    if not API_KEY:
        raise ValueError("WB_API_KEY is not found in .env")
    
    headers = {
        "Authorization": API_KEY
    }

    response = requests.get(
        endpoint,
        headers=headers,
        params=params,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
import sys
from pathlib import Path

project_root = "/Users/macbookair/Documents/wb_logistics_support"
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
from datetime import datetime, timedelta
from typing import Optional, List, Dict
import time
from src.ingestion.wb_client import fetch_report

In [ ]:
def get_coefficients(date_from: str, date_to: Optional[str] = None,
                         limit: int = 100000) -> List[Dict]:
        endpoint = "https://supplies-api.wildberries.ru/api/v1/acceptance/coefficients"
        params = {
            "dateFrom": date_from,
            "dateTo": date_to,
            "limit": limit
        } 
        return fetch_report(endpoint, params)

In [33]:
import json 

def get_warehouses() -> List[Dict]:
        endpoint = "https://supplies-api.wildberries.ru/api/v1/warehouses"
        params = {}
        return fetch_report(endpoint, params)
get = get_warehouses()
print(json.dumps(get, indent=2, ensure_ascii=False))
print(len(get))

[
  {
    "ID": 218987,
    "name": "Алматы Атакент",
    "address": "г.Алматы, ул. Тимирязева 42 павильон 7а",
    "workTime": "24/7",
    "isActive": true,
    "isTransitActive": false
  },
  {
    "ID": 204939,
    "name": "Астана",
    "address": "Нур-Султан, Шоссе Алаш, 69/1",
    "workTime": "24/7",
    "isActive": true,
    "isTransitActive": false
  },
  {
    "ID": 324108,
    "name": "Астана 2",
    "address": "Караганда, 91/2",
    "workTime": "24/7",
    "isActive": true,
    "isTransitActive": false
  },
  {
    "ID": 206236,
    "name": "Белые Столбы",
    "address": "Московская область, Домодедово, микрорайон Белые Столбы, владение Склады 104, с 4/1",
    "workTime": "24/7",
    "isActive": true,
    "isTransitActive": false
  },
  {
    "ID": 50172467,
    "name": "Владивосток СГТ",
    "address": "Центральный переулок, 3 С. Вольно-надеждинское, Приморский край",
    "workTime": "24/7",
    "isActive": true,
    "isTransitActive": false
  },
  {
    "ID": 301981,
    "n

In [ ]:
def create_warehouse_remains_task(**params) -> str:
    """
    Step 1: create report, return taskId.
    """
    endpoint = f"{BASE_URL_ANALYTICS}/warehouse_remains"
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        params=params,
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["data"]["taskId"]


def get_warehouse_remains_status(task_id: str) -> str:
    """
    Step 2: check status of the report.
    """
    endpoint = f"{BASE_URL_ANALYTICS}/warehouse_remains/tasks/{task_id}/status"
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["data"]["status"]


def download_warehouse_remains(task_id: str) -> List[Dict]:
    """
    Step 3: download final report data.
    """
    endpoint = f"{BASE_URL_ANALYTICS}/warehouse_remains/tasks/{task_id}/download"
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

In [ ]:
def get_warehouse_remains_report(
    poll_interval: int = 10,
    max_attempts: int = 30,
    **params,
) -> List[Dict]:
    """
    1) create report
    2) poll status until done
    3) download and return data (JSON list)
    """
    task_id = create_warehouse_remains_task(**params)
    print(f"Создан отчёт, taskId = {task_id}")

    for attempt in range(1, max_attempts + 1):
        time.sleep(poll_interval)
        status = get_warehouse_remains_status(task_id)
        print(f"[{attempt}/{max_attempts}] status = {status}")

        if status == "done":
            print("Отчёт готов, скачиваем...")
            break
    else:
        raise TimeoutError("Превышено время ожидания готовности отчёта")

    data = download_warehouse_remains(task_id)
    print(f"Получено строк: {len(data)}")
    return data

In [ ]:
params = {
    "locale": "ru",
    "groupByNm": True,
    # любые нужные groupBy*/filter* из доки WB
}

report = get_warehouse_remains_report(**params)
report[:3]

In [ ]:
from src.ingestion.tasks import generate_warehouse_remains_report

params = {
    "locale": "ru",
    "groupByNm": True,
    # другие groupBy*/filter* по необходимости
}

# отправляем задачу в Celery через Redis
async_result = generate_warehouse_remains_report.delay(**params)

# можно смотреть статус
async_result.state      # "PENDING" / "PROGRESS" / "SUCCESS" / "FAILURE"
async_result.info       # meta: попытка, статус WB, если PROGRESS

# когда хочешь результат (блокирует до завершения или таймаута)
result = async_result.get(timeout=600)
result
# -> {"rows": N, "filename": "warehouse_remains_....json"}

In [ ]:
def create_paid_storage_task(date_from: str, date_to: str) -> str:
    endpoint = f"{BASE_URL_ANALYTICS}/paid_storage"
    params = {
        "dateFrom": date_from,
        "dateTo": date_to
    }
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        params=params,
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["data"]["taskId"]


def get_paid_storage_status(task_id: str) -> str:
    endpoint = f"{BASE_URL_ANALYTICS}/paid_storage/tasks/{task_id}/status"
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["data"]["status"]


def download_paid_storage(task_id: str) -> List[Dict]:
    endpoint = f"{BASE_URL_ANALYTICS}/paid_storage/tasks/{task_id}/download"
    resp = requests.get(
        endpoint,
        headers={"Authorization": API_KEY},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

In [36]:
def get_paid_storage_report(
    date_from: str,
    date_to: str,
    poll_interval: int = 10,
    max_attempts: int = 30,
) -> List[Dict]:
    task_id = create_paid_storage_task(date_from, date_to)
    print(f"Создан отчёт, taskId = {task_id}")

    for attempt in range(1, max_attempts + 1):
        time.sleep(poll_interval)
        status = get_paid_storage_status(task_id)
        print(f"[{attempt}/{max_attempts}] status = {status}")

        if status == "done":
            print("Отчёт готов, скачиваем...")
            break
    else:
        raise TimeoutError("Превышено время ожидания готовности отчёта")

    data = download_paid_storage(task_id)
    print(f"Получено строк: {len(data)}")
    return data

paid = get_paid_storage_report(date_from="2026-03-01", date_to="2026-03-07")
print(f"paid_storage rows: {len(paid)}")
paid[:2]

Создан отчёт, taskId = d3ee81aa-e52c-4056-b375-f865a4a499e2
[1/30] status = done
Отчёт готов, скачиваем...
Получено строк: 704
paid_storage rows: 704


[{'date': '2026-03-02',
  'logWarehouseCoef': 0,
  'officeId': 120762,
  'warehouse': 'Электросталь',
  'warehouseCoef': 1.6,
  'giId': 32148137,
  'chrtId': 343395256,
  'size': 'M',
  'barcode': '2039614695201',
  'subject': 'Халаты домашние',
  'brand': 'Mari Style',
  'vendorCode': '29',
  'nmId': 215232550,
  'volume': 12.48,
  'calcType': 'короба: товары свыше базы',
  'warehousePrice': 1.59744,
  'barcodesCount': 1,
  'palletPlaceCode': 0,
  'palletCount': 0,
  'originalDate': '2026-03-02',
  'loyaltyDiscount': 0,
  'tariffFixDate': '2025-08-27',
  'tariffLowerDate': ''},
 {'date': '2026-03-02',
  'logWarehouseCoef': 0,
  'officeId': 301987,
  'warehouse': 'Сарапул',
  'warehouseCoef': 1.6,
  'giId': 32148137,
  'chrtId': 275891051,
  'size': 'XXL',
  'barcode': '2038049231855',
  'subject': 'Халаты домашние',
  'brand': 'Mari Style',
  'vendorCode': '10',
  'nmId': 165708460,
  'volume': 16.128,
  'calcType': 'короба: товары свыше базы',
  'warehousePrice': 2.064384,
  'barcode

In [ ]:
import requests
resp = requests.get(
    "https://seller-analytics-api.wildberries.ru/api/v1/paid_storage",
    headers={"Authorization": API_KEY},
    params={"dateFrom": "2026-03-01", "dateTo": "2026-03-07"}
)
print(resp.status_code)
print(resp.json())

200
{'data': {'taskId': '3cfbc190-5b93-49ff-abd7-f00a73fb334d'}}


In [40]:
def get_region_sale(date_from: str, date_to: str,
                    limit: int = 100000, offset: int = 0) -> List[Dict]:
    endpoint = f"{BASE_URL_ANALYTICS}/analytics/region-sale"
    params = {
        "dateFrom": date_from,
        "dateTo": date_to,
        "limit": limit,
        "offset": offset,
    }
    return fetch_report(endpoint, params)

region = get_region_sale(date_from="2026-03-01", date_to="2026-03-07")
print(f"Получено строк: {len(region)}")
print((region))
print(region.keys() if isinstance(region, dict) else region[:1])

# region[:3]

Получено строк: 1
{'report': [{'cityName': 'Владивосток', 'countryName': 'Россия', 'foName': 'Дальневосточный федеральный округ', 'nmID': 198435750, 'regionName': 'Приморский край', 'sa': '18', 'saleInvoiceCostPrice': 2308.66, 'saleInvoiceCostPricePerc': 14.6495922067159, 'saleItemInvoiceQty': 1}, {'cityName': 'Хабаровск', 'countryName': 'Россия', 'foName': 'Дальневосточный федеральный округ', 'nmID': 206765253, 'regionName': 'Хабаровский край', 'sa': '22', 'saleInvoiceCostPrice': 2334.37, 'saleInvoiceCostPricePerc': 14.8127349023206, 'saleItemInvoiceQty': 1}, {'cityName': 'Самара', 'countryName': 'Россия', 'foName': 'Приволжский федеральный округ', 'nmID': 482388068, 'regionName': 'Самарская область', 'sa': '112', 'saleInvoiceCostPrice': 2195.19, 'saleInvoiceCostPricePerc': 13.9295688045276, 'saleItemInvoiceQty': 1}, {'cityName': 'Северодвинск', 'countryName': 'Россия', 'foName': 'Северо-Западный федеральный округ', 'nmID': 198435750, 'regionName': 'Архангельская область', 'sa': '18',

In [43]:
def get_goods_return(date_from: str, date_to: str,
                     limit: int = 100000, offset: int = 0) -> List[Dict]:
    endpoint = f"{BASE_URL_ANALYTICS}/analytics/goods-return"
    params = {
        "dateFrom": date_from,
        "dateTo": date_to,
        "limit": limit,
        "offset": offset,
    }
    return fetch_report(endpoint, params)

returns = get_goods_return(date_from="2026-03-01", date_to="2026-03-31")
print(f"Получено строк: {len(returns)}")
print(returns)
# returns[:3]

Получено строк: 1
{'report': []}
